In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import sigpy as sp
import pandas as pd
import sigpy.plot as pl
import numpy as np
import os
import sys
from pathlib import Path
import jax as jx
import time

ksp = np.load(r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Sigpytutorial/projection_ksp.npy")
coord = np.load(r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Sigpytutorial/projection_coord.npy")

sys.path.insert(0,'/Users/ayman/Desktop/MSc Project Local/MSc-Project-ZTE')
import aymansigmri as asm


sys.path.insert(0, "/Users/ayman/Documents/GitHub/riesling/python")

import riesling as rlp



riesling_bin = Path.home() / "Documents" / "github" / "riesling" / "build" / "cxx" / "riesling"
os.environ["PATH"] += os.pathsep + str(riesling_bin)

In [ ]:
'''data_dir = r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/hires.h5"
data = r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/hires"'''

In [ ]:
'''data_dir = r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/"
raw = 'hires'
data=f'{data_dir}/{raw}'''
dataraw = r'/Users/ayman/Desktop/ZTE Data/raw0'

In [ ]:
data='hires'
!riesling downsamp --res=6,6,6 --trim {data}.h5 lores.h5 -v2

In [ ]:
data = 'hires'
!riesling sense-calib {data}.h5 {data}-ks.h5 -v2

In [ ]:
data='lores'
#!riesling sense-maps {data}-ks.h5 {data}.h5 {data}-sm.h5 -v2

In [ ]:
#rlp.plot.traj2d(f'{data1}.h5', sample=slice(0,-1,8), color='sample')

rlp.plot.traj2d(f'{data}.h5', sample=slice(0,-1,8), color='sample')

In [ ]:
display(rlp.plot.sense(f'{data}-sm.h5',rows=2,rotates=1))

In [ ]:
!riesling recon-lsq --precon-p=1 --sense="{data}-ks.h5" "{data}.h5" "{data}-lsq.h5" -v2


In [ ]:
!riesling op-nufft {data}.h5 {data}-nufft.h5 -v2

In [ ]:
rlp.plot.planes(f'{data}-nufft.h5', title='LSQ')

In [ ]:
rlp.plot.planes('hires-lsq.h5', title='LSQ - No WASPI')

In [ ]:
lowres = 'lores'
coorlowres = rlp.io.read_trajectory(f'{lowres}.h5')
coorlowres = np.asarray(coorlowres)
kspacelowres = rlp.io.read_data(f'{lowres}.h5')
kspacelowres = np.asarray(kspacelowres)
kspacelowres = kspacelowres[0, 0] 
kspacelowres = np.moveaxis(kspacelowres, -1, 0)



hires = 'hires'
coorhires = rlp.io.read_trajectory(f'{hires}.h5')
coorhires = np.asarray(coorhires)
kspacehires = rlp.io.read_data(f'{hires}.h5')
kspacehires = np.asarray(kspacehires)
kspacehires = kspacehires[0, 0] 
kspacehires = np.moveaxis(kspacehires, -1, 0)

In [ ]:
print(coorlowres.shape)
print(kspacelowres.shape)

print(coorhires.shape)
print(kspacehires.shape)

In [ ]:
data = 'hires'
#!riesling downsamp --res=6,6,6 --trim {data}.h5 lores.h5 -v2
#!riesling op-nufft {data}.h5 {data}-nufft.h5 -v2

rlp.plot.planes('hires-lsq.h5', title='LSQ - No WASPI')

In [ ]:
dcf_new = np.sqrt(coorlowres[...,0]**2 + coorlowres[...,1]**2 + coorlowres[...,2]**2)

zte_im_grid = sp.nufft_adjoint(kspacelowres* dcf_new, coorlowres)
gridded_data = sp.fft(zte_im_grid, axes=(-2, -1))
#zte_im = np.sum(np.abs(zte_im_grid)**2, axis=0)**0.5

In [ ]:
gridded_data.shape

## Goal of the notebook

### Integrate riesling into sigpy pipeline

1. Start with NUFFT inverse

- get the above file format into something that riseling can read

In [ ]:
def zeropadding3d(cart_kspace,resize_x, resize_y, resize_z):
    n_coils = cart_kspace.shape[0]
    image_grid = sp.ifft(cart_kspace)
    enlarged_image_grid = sp.resize(image_grid, [n_coils,resize_x, resize_y, resize_z])
    enlarged_cartesian_kspace = sp.fft(enlarged_image_grid, axes=(-2, -1))
    return enlarged_cartesian_kspace

In [ ]:
enlarged_cartesian = zeropadding3d(gridded_data, 160,160,160)

In [ ]:
import matplotlib
#all_coords = zte_radial_coords.reshape(-1,2)
#all_kspace = zte_radial_kspace[0].reshape(-1)

all_ztecoords= coorlowres.reshape(-1,3)
all_zteksp = kspacelowres.reshape(-1)


fig = plt.figure(figsize=(15,15))

# --- Subplot 1: full 3D trajectory ---
ax0 = fig.add_subplot(2, 2, 1, projection='3d')
im0 = ax0.scatter(
    all_ztecoords[:, 0], all_ztecoords[:, 1], all_ztecoords[:, 2],
    c=np.abs(kspacelowres[0]).reshape(-1),
    s=1
)
ax0.set_title('All spokes 1 coil (3D)')
ax0.set_xlabel('kx')
ax0.set_ylabel('ky')
ax0.set_zlabel('kz')
ax0.set_box_aspect((1, 1, 1))
fig.colorbar(im0, ax=ax0, label='k-space', shrink=0.6)

# --- Subplot 2: slice near kx = 0 (2D view of ky-kz plane) ---
ax1 = fig.add_subplot(2, 2, 2)  

tol = 0.5  # slab half-thickness; adjust to your coordinate units
mask = np.abs(all_ztecoords[:, 0]) < tol

im1 = ax1.scatter(
    all_ztecoords[mask, 1], all_ztecoords[mask, 2],
    c=np.abs(kspacelowres[0]).reshape(-1)[mask],
    s=1
)
ax1.set_title(f'Slice at kx approx 0 (kx < {tol}), {mask.sum()} points')
ax1.set_xlabel('ky')
ax1.set_ylabel('kz')
ax1.set_aspect('equal')
ax1.xaxis.set_major_locator(plt.MultipleLocator(1))
ax1.yaxis.set_major_locator(plt.MultipleLocator(1))
ax1.grid(visible=True, which='minor', linewidth=1)
ax1.xaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax1.yaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax1.set_xlim(-10,10)
ax1.set_ylim(-10,10)
#ax1.axis('equal')
fig.colorbar(im1, ax=ax1, label='k-space')

ax2 = fig.add_subplot(2, 2, 3)

grid_vol = np.abs(gridded_data[0])
kx_centre = grid_vol.shape[0] // 2  # kx = 0 lives at the centre index

im2 = ax2.imshow(
    grid_vol[kx_centre],
    origin='lower',
    norm=matplotlib.colors.LogNorm()  # k-space dynamic range is huge
)
ax2.set_title('Zeropadded cartesian k-space @ kx = 0')
ax2.set_xlabel('kz')
ax2.set_ylabel('ky')
fig.colorbar(im2, ax=ax2, label='k-space')

ax2 = fig.add_subplot(2, 2, 4)

grid_vol_zeropadding = np.abs(enlarged_cartesian[0])
kx_centre_large = grid_vol_zeropadding.shape[0] // 2  # kx = 0 lives at the centre index

im2 = ax2.imshow(
    grid_vol_zeropadding[kx_centre_large],
    origin='lower',
    norm=matplotlib.colors.LogNorm()  # k-space dynamic range is huge
)
ax2.set_title('Zeropadded cartesian k-space @ kx = 0')
ax2.set_xlabel('kz')
ax2.set_ylabel('ky')
fig.colorbar(im2, ax=ax2, label='k-space')
plt.show()


In [ ]:
'''#all_coords = zte_radial_coords.reshape(-1,2)
#all_kspace = zte_radial_kspace[0].reshape(-1)




fig = plt.figure(figsize=(15,15))

# --- Subplot 1: full 3D trajectory ---
ax0 = fig.add_subplot(2, 2, 1, projection='3d')

half = all_ztecoords[:, 1] >= 0   # keep only ky >= 0

im0 = ax0.scatter(
    all_ztecoords[half, 0], all_ztecoords[half, 1], all_ztecoords[half, 2],
    c=np.abs(kspacelowres[0]).reshape(-1)[half],
    s=1
)
ax0.set_title('All spokes 1 coil (3D), ky >= 0 cutaway')
ax0.set_xlabel('kx')
ax0.set_ylabel('ky')
ax0.set_zlabel('kz')
ax0.set_box_aspect((1, 1, 1))
fig.colorbar(im0, ax=ax0, label='k-space', shrink=0.6)



# --- Subplot 2: slice near kx = 0 (2D view of ky-kz plane) ---
ax1 = fig.add_subplot(2, 2, 2)  

tol = 0.5  # slab half-thickness; adjust to your coordinate units
mask = np.abs(all_ztecoords[:, 0]) < tol

im1 = ax1.scatter(
    all_ztecoords[mask, 1], all_ztecoords[mask, 2],
    c=np.abs(kspacelowres[0]).reshape(-1)[mask],
    s=1
)
ax1.set_title(f'Slice at kx approx 0 (kx < {tol}), {mask.sum()} points')
ax1.set_xlabel('ky')
ax1.set_ylabel('kz')
ax1.set_aspect('equal')
ax1.xaxis.set_major_locator(plt.MultipleLocator(1))
ax1.yaxis.set_major_locator(plt.MultipleLocator(1))
ax1.grid(visible=True, which='minor', linewidth=1)
ax1.xaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax1.yaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax1.set_xlim(-10,10)
ax1.set_ylim(-10,10)
#ax1.axis('equal')
fig.colorbar(im1, ax=ax1, label='k-space')

ax2 = fig.add_subplot(2, 2, 3)

grid_vol = np.abs(gridded_data[0])
kx_centre = grid_vol.shape[0] // 2  # kx = 0 lives at the centre index

im2 = ax2.imshow(
    grid_vol[kx_centre],
    origin='lower',
    norm=matplotlib.colors.LogNorm()
)
ax2.set_title('Zeropadded cartesian k-space @ kx = 0')
ax2.set_xlabel('kz')
ax2.set_ylabel('ky')
fig.colorbar(im2, ax=ax2, label='k-space')

ax2 = fig.add_subplot(2, 2, 4)

grid_vol_zeropadding = np.abs(enlarged_cartesian[0])
kx_centre_large = grid_vol_zeropadding.shape[0] // 2  # kx = 0 lives at the centre index

im2 = ax2.imshow(
    grid_vol_zeropadding[kx_centre_large],
    origin='lower',
    norm=matplotlib.colors.LogNorm()  # k-space dynamic range is huge
)
ax2.set_title('Zeropadded cartesian k-space @ kx = 0')
ax2.set_xlabel('kz')
ax2.set_ylabel('ky')
ax2.set_xlim(30,130)
ax2.set_ylim(30,130)
fig.colorbar(im2, ax=ax2, label='k-space')
#plt.savefig('zte_data_panel')
plt.show()
'''

In [ ]:
width=4
gap = 2
num_iters = 50

resize_wid=160
rw_div2 = int(resize_wid/2)

inner_wid=100
indiv2 = int(inner_wid/2)

In [ ]:
def inner_portion(enlarged_kspace, inner_sidelen):
    """Take an inner square of kspace to speed up the hankel loop

        Args:
            enlarged_kspace: Zero padded kspace in cartesian coordinates (n_coils, nx, ny)
            inner_sidelen: Side legnth of the inner square, divided by 2 to find from central point outward

        Returns:
            isolated_kspace: Inner kspace array
            isolation_mask: Mask of the inner region to use when 'jigsawing' the transformed array back in
            Start: xy index of the enlarged array that the inner region starts at
            End: xy index of the enlarged array that the inner region ends at
    """
    cx, cy, cz = enlarged_kspace.shape[1] // 2, enlarged_kspace.shape[2] // 2, enlarged_kspace.shape[3] // 2
    N = inner_sidelen/2
    start, end = int(cx-N), int(cx+N)
    isolation_mask = np.ones(enlarged_kspace.shape[1:], dtype=int)
    isolation_mask[start:end, start:end, start:end] = 0
    isolated_kspace = enlarged_kspace[:, start:end, start:end, start:end]
    
    return(isolated_kspace, isolation_mask, start, end)

inner_region, inner_mask, start, end = inner_portion(enlarged_kspace=enlarged_cartesian, inner_sidelen=inner_wid)

In [ ]:
width=4
gap = 3
num_iters = 50

resize_wid=160
rw_div2 = int(resize_wid/2)

inner_wid=100
indiv2 = int(inner_wid/2)

In [ ]:
grid_vol_zeropadding = np.abs(enlarged_cartesian[0])
kx_centre_large = grid_vol_zeropadding.shape[0] // 2


cy = cx = cz = inner_wid // 2
r = 5  # radius in pixels; adjust to cover the gap
yy, xx, zz = np.ogrid[:inner_wid, :inner_wid, :inner_wid]
mask = (yy - cy)**2 + (xx - cx)**2 + (zz - cz)**2 > r**2 + 2

mask[(cy-6):(cy+7), cx, cz] = False   # line along axis 0
mask[cy, (cx-6):(cx+7), cz] = False   # line along axis 1
mask[cy, cx, (cz-6):(cz+7)] = False


'''masked_inner = inner_region.copy()
masked_inner[:,~mask] = 0

inner_removed = inner_region.copy()
inner_removed[:,mask] = 0

totalwid = 1024
totaldiv = totalwid // 2

mask2 = np.ones((totalwid, totalwid), dtype=bool)

mask2[(indiv2-3):(indiv2+3), indiv2:indiv2+1] = False
mask2[(indiv2), (indiv2-2):(indiv2+3)] = False'''


#asm.plot_mask(inner_region, mask=mask, zte_radial_coords=all_ztecoords, zte_radial_kspace=kspacelowres, innersidelen=inner_wid,sidelencart=10,sidelenrad=4)

isolated_kspace = inner_region
innersidelen=inner_wid
sidelencart = 15
zte_radial_kspace = kspacelowres
sidelenrad = 10

In [ ]:
import matplotlib.pyplot as plt

pad = r + 8                     # a bit bigger than the sphere + arm length
sl = slice(cy - pad, cy + pad + 1)
region = ~mask[sl, sl, sl]      # True where samples are MASKED OUT (sphere + lines)

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')
ax.voxels(region, facecolors='crimson', edgecolor='k', linewidth=0.2, alpha=0.8)
ax.set_xlabel('axis 0 (ky)')
ax.set_ylabel('axis 1 (kx)')
ax.set_zlabel('axis 2 (kz)')
ax.set_title(f'Masked region (sphere r={r} + axis lines), centre crop')
ax.set_box_aspect((1, 1, 1))
plt.show()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(30, 8))

grid_vol_zeropadding = np.abs(inner_region[0])
kx_centre_large = grid_vol_zeropadding.shape[0] // 2


im = ax[0].imshow(
    np.abs(grid_vol_zeropadding[kx_centre_large]),
    origin='lower',
    norm=matplotlib.colors.LogNorm(),
)
fig.colorbar(im, ax=ax[0], label='kspace value')
ax[0].set_title('Cartesian')
ax[0].set_xlabel('kz')
ax[0].set_ylabel('ky')
ax[0].set_xlim(innersidelen/2 - sidelencart,innersidelen/2 + sidelencart)
ax[0].set_ylim(innersidelen/2 - sidelencart,innersidelen/2 + sidelencart)
ax[0].xaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax[0].yaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax[0].grid(visible=True, which='minor', linewidth=1)
ax[0].xaxis.set_major_locator(plt.MultipleLocator(1))
ax[0].yaxis.set_major_locator(plt.MultipleLocator(1))


mask2d = mask[:, :, cz]          # central slice; pick the axis/index you care about

yy, xx = np.mgrid[0:mask2d.shape[0], 0:mask2d.shape[1]]
im1 = ax[1].scatter(xx.ravel(), yy.ravel(), c=mask2d.ravel().astype(int),
                    cmap='viridis', s=300, marker='s', zorder=3)
fig.colorbar(im1, ax=ax[1], label='mask')
ax[1].set_title(f'Centre mask @ kz = {cz}')
ax[1].set_xlabel('x')
ax[1].set_ylabel('y')
ax[1].axis('equal')

c = mask2d.shape[0] / 2
ax[1].set_xlim(c - sidelencart, c + sidelencart)
ax[1].set_ylim(c - sidelencart, c + sidelencart)

ax[1].xaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax[1].yaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax[1].grid(visible=True, which='minor', linewidth=1)
ax[1].xaxis.set_major_locator(plt.MultipleLocator(1))
ax[1].yaxis.set_major_locator(plt.MultipleLocator(1))



tol = 0.5  # slab half-thickness; adjust to your coordinate units
mask = np.abs(all_ztecoords[:, 0]) < tol

im2 = ax[2].scatter(all_ztecoords[mask, 1], all_ztecoords[mask, 2],c=np.abs(kspacelowres[0]).reshape(-1)[mask],s=1)
fig.colorbar(im2, ax=ax[2], label='kspace value')
ax[2].set_title('Radial')
ax[2].set_xlabel('kx')
ax[2].set_ylabel('ky')
ax[2].axis('equal')
ax[2].xaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax[2].yaxis.set_minor_locator(plt.MultipleLocator(1, offset=0.5))
ax[2].grid(visible=True, which='minor', linewidth=1)
ax[2].xaxis.set_major_locator(plt.MultipleLocator(1))
ax[2].yaxis.set_major_locator(plt.MultipleLocator(1))
ax[2].set_xlim(-10,10)
ax[2].set_ylim(-10,10)

plt.show()

In [ ]:
def hankel(kspace, w):
    # kspace: (N_c, X, Y, Z)
    N_c = kspace.shape[0]
    N_x = kspace.shape[1] - w + 1
    N_y = kspace.shape[2] - w + 1
    N_z = kspace.shape[3] - w + 1
    data_matrix = []
    for c in range(N_c):
        onecoil_matrix = np.empty((w*w*w, N_x*N_y*N_z), dtype=np.complex64)
        col_index = 0
        for k in range(N_z):        # z outermost
            for i in range(N_y):
                for j in range(N_x):    # x innermost, matching your 2D ordering
                    mat = kspace[c, j:j+w, i:i+w, k:k+w]
                    onecoil_matrix[:, col_index] = mat.reshape(-1, order='F')
                    col_index += 1
        data_matrix.append(onecoil_matrix)
    return np.vstack(data_matrix), N_c, N_x, N_y, N_z


def hankel_H_averaged(data_matrix, n_coils, Nx, Ny, Nz, w=3):
    kspace = []
    split_arr = np.array(np.split(data_matrix, n_coils))

    n_win_x = Nx - w + 1
    n_win_y = Ny - w + 1
    n_win_z = Nz - w + 1

    for c in range(n_coils):
        singlecoil = split_arr[c]
        transposed = singlecoil.T          # one row per window

        recon = np.zeros((Nx, Ny, Nz), dtype=np.complex64)
        counts = np.zeros((Nx, Ny, Nz), dtype=np.float32)

        for index in range(len(transposed)):
            win = transposed[index].reshape((w, w, w), order='F')

            # invert col_index = k*(n_win_y*n_win_x) + i*n_win_x + j
            j = index % n_win_x                      # x offset (innermost)
            i = (index // n_win_x) % n_win_y         # y offset
            k = index // (n_win_x * n_win_y)         # z offset (outermost)

            recon[j:j+w, i:i+w, k:k+w] += win
            counts[j:j+w, i:i+w, k:k+w] += 1

        kspace.append(recon / counts)
    return np.array(kspace)

In [ ]:
def softimpute_ALS_time(X_H, M_H, rank, lamda, n_iters):
    I = np.eye(rank)
    m, n = np.shape(X_H)
    U = np.random.randn(m, rank) + 1j * np.random.randn(m, rank)
    V = np.random.randn(n, rank) + 1j * np.random.randn(n, rank)

    D = I.copy()

    A = np.dot(U, D)
    B = np.dot(V, D)
    iter_count = 0
    ABt = A @ B.conj().T

    iter_times = []
    t_total_start = time.perf_counter()

    while iter_count < n_iters:
        t_start = time.perf_counter()

        X_star = np.where(M_H, X_H, ABt)
        X_star1H = X_star.copy()
        A = X_star @ B @ np.linalg.inv(B.conj().T @ B + lamda*I)
        ABt = A @ B.conj().T
        X_star = np.where(M_H, X_H, ABt)
        B = X_star.conj().T @ A @ np.linalg.inv(A.conj().T @ A + lamda*I)
        ABt = A @ B.conj().T
        iter_count += 1

        t_elapsed = time.perf_counter() - t_start
        iter_times.append(t_elapsed)

    t_total = time.perf_counter() - t_total_start
    t_mean = np.mean(iter_times)

    return (ABt, X_star1H, t_total, t_mean, iter_times)



def LORAKS_loop_timing(n_iters, window_size, zero_thresh, cartesian_inputkspace, zeropadded,dtg_mask, im_dim):
    ksp_forhankel = cartesian_inputkspace.copy()
    iter_count = 0
    deltas = []
    k_prev = None
    timings = {"hankel": [], "svd": [], "unlift": [], "consistency": [], "iter_total": [], "total": []}
    hankel_size = []
    t_loop_start = time.perf_counter()
    while iter_count < n_iters:
        t0 = time.perf_counter()

        hankel_matrix, n_coils, Numx, Numy = hankel(kspace=ksp_forhankel, w=window_size)
        t1 = time.perf_counter()

        U, S_reduced, Vh = asm.sig_val_thresholding_jax(data=hankel_matrix, zero_thresh=zero_thresh)
        data_recon = (U * S_reduced) @ Vh
        data_recon = jx.block_until_ready(data_recon)
        t2 = time.perf_counter()

        kspace_cart_coils_recon = hankel_H_averaged(
            data_recon, n_coils=ksp_forhankel.shape[0],
            Nx=ksp_forhankel.shape[1], Ny=ksp_forhankel.shape[2],
            w=window_size)
        t3 = time.perf_counter()

        kspace_cart_coils_consistent = np.where(dtg_mask, cartesian_inputkspace, kspace_cart_coils_recon)
        ksp_forhankel = np.asarray(kspace_cart_coils_consistent)  # ensure NumPy, on-host
        t4 = time.perf_counter()

        vec = ksp_forhankel[:, ~dtg_mask]
        if k_prev is not None:
            deltas.append(np.linalg.norm(vec - k_prev) / np.linalg.norm(k_prev))
        k_prev = vec.copy()

        timings["hankel"].append(t1 - t0)
        timings["svd"].append(t2 - t1)
        timings["unlift"].append(t3 - t2)
        timings["consistency"].append(t4 - t3)
        timings["iter_total"].append(t4 - t0)
        hankel_size.append(hankel_matrix.shape)

        iter_count += 1
    t_loop_end = time.perf_counter()
    timings["total"].append(t_loop_end - t_loop_start)
    output_kspace = ksp_forhankel.copy()
    filled_ksp = asm.rebuild(output_kspace=output_kspace, inner_mask=inner_mask,inner_start=start, inner_end=end,enlarged_kspace=zeropadded,resize_x=im_dim, resize_y=im_dim)
    im_grid0 = sp.ifft(filled_ksp, axes=(-2, -1))
    im_0 = np.sum(np.abs(im_grid0)**2, axis=0)**0.5

    return (im_0, filled_ksp, deltas, timings, hankel_size)

def LORAKS_imputeals(n_iters, window_size, cartesian_inputkspace, dtg_mask, zeropadded,rank, lamda, im_dim):
    ksp_forhankel = cartesian_inputkspace.copy()
    ksp_zerod = ksp_forhankel * dtg_mask
    hankel_matrix, n_coils, Numx, Numy = hankel(kspace=ksp_zerod, w=window_size)
    mask_coiled = np.broadcast_to(dtg_mask, (cartesian_inputkspace.shape))
    masked_hankel, *_ = hankel(kspace=mask_coiled, w=window_size)
    masked_hankel = np.real(masked_hankel) > 0.5
    
    
    filled_hankel, X_star1, t_total, t_mean, iter_times = softimpute_ALS_time(X_H = hankel_matrix, M_H = masked_hankel, rank = rank, lamda = lamda, n_iters=n_iters)

    kspace_cart_coils_recon = hankel_H_averaged(filled_hankel, n_coils=ksp_forhankel.shape[0], Nx=ksp_forhankel.shape[1], Ny=ksp_forhankel.shape[2], w=window_size)
    kspace_cart_coils_recon = np.where(dtg_mask, cartesian_inputkspace, kspace_cart_coils_recon)
    output_kspace = kspace_cart_coils_recon.copy()
    filled_ksp = asm.rebuild(output_kspace=output_kspace, inner_mask = inner_mask, inner_start = start, inner_end = end, enlarged_kspace=zeropadded, resize_x = im_dim, resize_y = im_dim)
    im_grid0 = sp.ifft(filled_ksp, axes=(-2, -1))
    im_0 = np.sum(np.abs(im_grid0)**2, axis=0)**0.5

    return (im_0, filled_ksp, masked_hankel, t_total, t_mean, iter_times)



def LORAKS_loop(n_iters, window_size, zero_thresh, cartesian_inputkspace, zeropadded,dtg_mask, im_dim):
    ksp_forhankel = cartesian_inputkspace.copy()
    iter_count = 0
    deltas = []
    k_prev = None
    while iter_count < n_iters:
        hankel_matrix, n_coils, Numx, Numy = hankel(kspace=ksp_forhankel, w=window_size)

        U, S_reduced, Vh = asm.sig_val_thresholding_jax(data=hankel_matrix, zero_thresh=zero_thresh)
        data_recon = (U * S_reduced) @ Vh

        kspace_cart_coils_recon = hankel_H_averaged(data_recon, n_coils=ksp_forhankel.shape[0], Nx=ksp_forhankel.shape[1], Ny=ksp_forhankel.shape[2], w=window_size)
        kspace_cart_coils_consistent = kspace_cart_coils_recon.copy()

        kspace_cart_coils_recon = np.where(dtg_mask, cartesian_inputkspace, kspace_cart_coils_recon)
        ksp_forhankel = kspace_cart_coils_consistent   

        vec = ksp_forhankel[:, ~dtg_mask]

        if k_prev is not None:
            deltas.append(np.linalg.norm(vec - k_prev) / np.linalg.norm(k_prev))
        k_prev = vec.copy()
        iter_count += 1

    output_kspace = ksp_forhankel.copy()
    filled_ksp = asm.rebuild(output_kspace=output_kspace, inner_mask = inner_mask, inner_start = start, inner_end = end, enlarged_kspace=zeropadded, resize_x = im_dim, resize_y = im_dim)
    im_grid0 = sp.ifft(filled_ksp, axes=(-2, -1))
    im_0 = np.sum(np.abs(im_grid0)**2, axis=0)**0.5
    
    return(im_0, filled_ksp, deltas)

In [ ]:
import time

timings = {'als': [], 'svd': []}
t0 = time.perf_counter()
im_als, output_kspaceals, zeroed, total_time, mean_time, raw_times = LORAKS_imputeals(n_iters=2, window_size=10, cartesian_inputkspace=inner_region, zeropadded = enlarged_cartesian,dtg_mask=mask, rank= 10, lamda = 5*10**-5, im_dim=256)
t1 = time.perf_counter()
timings['als'].append(t1 - t0)

print(f"Als done in {t1-t0}")

t2 = time.perf_counter()
ims_svd, output_kspacesvd, deltas, svdtimes, hankelshape = LORAKS_loop_timing(n_iters=2, window_size=5, zero_thresh=0.2,cartesian_inputkspace=inner_region, dtg_mask=mask, im_dim=256)
t3 = time.perf_counter()
#timings['svd'].append(t3-t2)

print(f"SVD done in {t3-t2}")

svdtimes_cleaned = {key: float(np.mean(times[1:])) for key, times in svdtimes.items()}

print(svdtimes_cleaned)
print(mean_time)

test_results = {'2 points removed':preloop, 'als, window:10 ,rank:10, lam:5e-5':im_als, 'svd, window:5, zerthresh:0.2': ims_svd,'nogap':img_nogap}

asm.diff_matrix(test_results, title=f'100 Iterations, Stride of 1.')